In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
from tensorflow.keras.utils import to_categorical

# 1. MNIST 데이터 로드
mnist = keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# 2. 데이터 정규화 (0~255 사이의 값을 0~1 사이로 변환)
x_train = train_images / 255.0
x_test = test_images / 255.0

# 3. 레이블 전처리 (One-hot Encoding)
NUM_CLASSES = 10
class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
y_train = to_categorical(train_labels, NUM_CLASSES)
y_test = to_categorical(test_labels, NUM_CLASSES)

print("converted y.shape =", y_train.shape)

In [ ]:
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model

# 방식 A: 함수형 API (유연한 설계 가능)
input_layer = Input(shape=(28, 28))
x = Flatten()(input_layer)
x = Dense(128, activation='relu')(x)
output_layer = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(input_layer, output_layer)

# 방식 B: 순차형 API (직관적이고 간결함)
# model = keras.Sequential([
#     Flatten(input_shape=(28, 28)),
#     Dense(128, activation='relu'),
#     Dense(10, activation='softmax')
# ])

# 모델 구조 시각화 (파일로 저장)
from keras.utils import plot_model
plot_model(model, to_file='model.png', show_shapes=True)

In [ ]:
# 예측 이미지와 레이블을 그리는 헬퍼 함수
def plot_image(i, predictions_array, true_label, img):
    predictions_array, true_label, img = predictions_array[i], true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(img, cmap=plt.cm.binary)

    predicted_label = np.argmax(predictions_array)
    color = 'blue' if predicted_label == np.argmax(true_label) else 'red'
    
    plt.xlabel("{} {:2.0f}% ({})".format(class_names[predicted_label],
                                        100*np.max(predictions_array),
                                        class_names[np.argmax(true_label)]),
                                        color=color)

def plot_value_array(i, predictions_array, true_label):
    predictions_array, true_label = predictions_array[i], true_label[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    thisplot = plt.bar(range(10), predictions_array, color='#777777')
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array)
    thisplot[predicted_label].set_color('red')
    thisplot[np.argmax(true_label)].set_color('blue')

# 실제로 여러 장의 예측 결과 출력하기
predictions = model.predict(x_test)
num_rows = 5
num_cols = 3
num_images = num_rows * num_cols
plt.figure(figsize=(2 * 2 * num_cols, 2 * num_rows))

for i in range(num_images):
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 1)
    plot_image(i + 720, predictions, y_test, x_test)
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 2)
    plot_value_array(i + 720, predictions, y_test)
plt.show()